# Pandas: Raw DataFrames

So far we've loaded data from neatly formatted files. In the real world, data often arrives **raw**: as a Python dictionary, as a headerless CSV, or with missing values scattered all over.

This lesson covers two practical workflows:

1. **Building a DataFrame from scratch** (from a Python dictionary).
2. **Loading and cleaning a messy DataFrame** (set column names, handle `NaN`, drop columns/rows, reset the index).


In [2]:
import pandas as pd

## Part 1 — Build a DataFrame from a Dictionary

A `DataFrame` can be constructed directly from a Python `dict`, where:

- each **key** becomes a **column name**
- each **value** is a list/array of values for that column

This is the fastest way to create small, hand-crafted datasets — useful for demos, tests, and quick experiments.

In [3]:
raw_data = {
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'age': [25, 30, 35, 28],
    'city': ['Cairo', 'Riyadh', 'Dubai', 'Amman'],
    'active': [True, False, True, True],
}

df = pd.DataFrame(raw_data)
df

,name,age,city,active
0,Alice,25,Cairo,True
1,Bob,30,Riyadh,False
2,Charlie,35,Dubai,True
3,Diana,28,Amman,True


### Reorder columns

By default, columns appear in the order the dictionary keys were inserted. To present them in a specific order, **index the DataFrame with a list of column names**:

In [4]:
df = df[['name', 'city', 'age', 'active']]
df

,name,city,age,active
0,Alice,Cairo,25,True
1,Bob,Riyadh,30,False
2,Charlie,Dubai,35,True
3,Diana,Amman,28,True


### Add a new column

Assigning a list (or `Series`) to a new column name creates the column:

In [5]:
df['country'] = ['Egypt', 'Saudi Arabia', 'UAE', 'Jordan']
df

,name,city,age,active,country
0,Alice,Cairo,25,True,Egypt
1,Bob,Riyadh,30,False,Saudi Arabia
2,Charlie,Dubai,35,True,UAE
3,Diana,Amman,28,True,Jordan


### Inspect column types

Pandas **infers** a dtype for each column based on its values. Check them with `.dtypes`:

In [6]:
df.dtypes

name         str
city         str
age        int64
active      bool
country      str
dtype: object

Strings show up as `object`, integers as `int64`, booleans as `bool`, and so on.

## Part 2 — Load and Clean a Raw CSV

Real datasets are rarely tidy. Many CSV files come with **no header row**, and once loaded they may contain **missing values** (`NaN`) you have to deal with.

We'll use the classic **Iris dataset** to walk through:

- reading a CSV that has no header
- naming the columns ourselves
- detecting and filling missing values
- dropping columns and rows
- resetting the index

In [7]:
import numpy as np

### Read a CSV with no header and set column names

By default, `pd.read_csv` treats the first row as the header.

In [16]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data'

iris = pd.read_csv(url)
iris.head(3)

,5.1,3.5,1.4,0.2,Iris-setosa
0,4.9,3.0,1.4,0.2,Iris-setosa
1,4.7,3.2,1.3,0.2,Iris-setosa
2,4.6,3.1,1.5,0.2,Iris-setosa


When the file has **no header**, pass `header=None` and provide the column names yourself with `names=[...]`:

In [8]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data'

iris = pd.read_csv(
    url,
    header=None,
    names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class'],
)
iris.head()

,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### Detect missing values

`isna()` (alias `isnull()`) returns a boolean DataFrame of the same shape, with `True` wherever a value is `NaN`. Combine it with `.sum()` to count missing values **per column**, or `.any().any()` to ask the global question *"are there any missing values at all?"*:

In [9]:
iris.isna().sum()

sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
class           0
dtype: int64

The dataset is clean — no `NaN` values yet. Let's **introduce some**, so we can practice cleaning them.

### Set specific cells to NaN

Use `.loc[rows, columns]` to assign `np.nan` to a slice of the DataFrame. Here we corrupt rows `10` through `29` of the `petal_length` column:

In [10]:
iris.loc[10:29, 'petal_length'] = np.nan
iris.isna().sum()

sepal_length     0
sepal_width      0
petal_length    20
petal_width      0
class            0
dtype: int64

> Note: with `.loc`, the **end label is inclusive** — `10:29` covers 20 rows (10, 11, …, 29).

### Fill missing values

`fillna(value)` replaces every `NaN` with `value`. You can fill with a constant, the column **mean**, **median**, the previous value (`method='ffill'`), etc. Here we fill with `1.0`:

In [11]:
iris['petal_length'] = iris['petal_length'].fillna(1.0)
iris.isna().sum()

sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
class           0
dtype: int64

### Drop a column

Use `df.drop(columns=[...])` to remove one or more columns. By default this returns a **new** DataFrame, so you must reassign (or pass `inplace=True`):

In [12]:
iris = iris.drop(columns=['class'])
iris.head()

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


### Drop rows containing NaN

Let's deliberately set the **first 3 rows** to `NaN`:

In [13]:
iris.iloc[0:3, :] = np.nan
iris.head()

,sepal_length,sepal_width,petal_length,petal_width
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


Now use `dropna()` to remove **every row that contains at least one `NaN`**:

In [14]:
iris = iris.dropna()
iris.head()

,sepal_length,sepal_width,petal_length,petal_width
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
5,5.4,3.9,1.7,0.4
6,4.6,3.4,1.4,0.3
7,5.0,3.4,1.5,0.2


### Reset the index

After dropping rows, the **index becomes non-contiguous** (notice it starts at `3` now, not `0`). `reset_index(drop=True)` renumbers the index starting from `0`, throwing away the old one:

In [15]:
iris = iris.reset_index(drop=True)
iris.head()

,sepal_length,sepal_width,petal_length,petal_width
0,4.6,3.1,1.5,0.2
1,5.0,3.6,1.4,0.2
2,5.4,3.9,1.7,0.4
3,4.6,3.4,1.4,0.3
4,5.0,3.4,1.5,0.2


> Without `drop=True`, the old index is kept as a new column called `index`.

## Key Takeaways

**Building DataFrames from scratch**

- `pd.DataFrame(dict)` — each key becomes a column.
- Reorder columns with `df[['col_a', 'col_b', ...]]`.
- Add a new column by assignment: `df['new_col'] = [...]`.
- Inspect types with `df.dtypes`.

**Loading and cleaning raw data**

- `pd.read_csv(url, header=None, names=[...])` — read a headerless CSV and name the columns yourself.
- `df.isna().sum()` — count missing values per column.
- `df.loc[rows, cols] = np.nan` — assign `NaN` to specific cells.
- `df['col'].fillna(value)` — replace `NaN` with a value (constant, mean, median, …).
- `df.drop(columns=[...])` — remove columns.
- `df.dropna()` — drop rows containing `NaN`.
- `df.reset_index(drop=True)` — renumber the index after dropping rows.